# 🧼 Lab W3-2 — Dirty Data Gauntlet

**สัปดาห์ที่ 3 — Data Management & Data Warehouse**

ใช้คู่กับสื่อจำลอง **Dirty Data Gauntlet** (`/sims/dirty-data`)

## สิ่งที่จะได้เรียนรู้
1. เปลี่ยนความคาดหวังด้านคุณภาพข้อมูลให้เป็น **กฎที่ทดสอบได้** ครบ 6 มิติ
2. แยกแถวที่ไม่ผ่านไปตาราง `etl_rejects` พร้อมเหตุผล แทนการทิ้งเงียบๆ
3. สร้าง `etl_audit` และพิสูจน์การ **กระทบยอด (reconciliation)**

## ข้อมูล
`sales_raw_dirty.csv` — ยอดขายจาก 3 ระบบต้นทาง (POS · Mobile App · Marketplace)

In [ ]:
import pandas as pd

BASE = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
        "master/datasets/week03/")
raw = pd.read_csv(BASE + "sales_raw_dirty.csv")
ref_product = pd.read_csv(BASE + "ref_product.csv")
ref_store = pd.read_csv(BASE + "ref_store.csv")

print(f"แถวข้อมูลดิบ : {len(raw):,}")
print(f"ระบบต้นทาง   : {raw.source_system.unique().tolist()}")
raw.head()

## ส่วนที่ 1 — Data Profiling ก่อนแตะข้อมูล

กฎเหล็กของงานคลังข้อมูล: **ห้ามแก้ข้อมูลก่อนเข้าใจว่าทำไมมันถึงเป็นแบบนั้น**

In [ ]:
print("=== รูปแบบรหัสสินค้าที่พบ แยกตามระบบต้นทาง ===")
print(pd.crosstab(raw.source_system, raw.sku.str.match(r"^P-\d{3}$")).to_string())
print("\n=== ตัวอย่างค่า sku ที่ไม่ตรงมาตรฐาน ===")
print(raw.loc[~raw.sku.str.match(r"^P-\d{3}$"), ["source_system", "sku"]].drop_duplicates().head(10).to_string(index=False))
print("\n=== รูปแบบวันที่ ===")
print(pd.crosstab(raw.source_system, raw.txn_date.str.contains("/")).to_string())

### 🧑‍💻 งานที่ 1 — สำรวจให้ครบ
ตรวจและรายงานจำนวนของแต่ละอาการต่อไปนี้ **ก่อน** ทำความสะอาด

1. `txn_id` ซ้ำ
2. `customer_id` ที่ว่าง หรือเป็น `NULL` / `N/A`
3. `store_id` ที่มีช่องว่างนำหน้า-ต่อท้าย หรือเป็นตัวพิมพ์เล็ก
4. แถวที่ `net_amount` ไม่ตรงกับ `quantity × unit_price − discount`
5. `store_id` ที่ไม่มีใน `ref_store`
6. รายการคืนสินค้า — มีกี่วิธีบันทึกในไฟล์นี้

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 2 — เขียนกฎคุณภาพให้ครบ 6 มิติ

| มิติ | คำถามที่กฎต้องตอบ |
|---|---|
| Completeness | ค่าที่จำเป็นครบหรือไม่ |
| Uniqueness | มีการนับซ้ำหรือไม่ |
| Validity | อยู่ในรูปแบบและช่วงที่ยอมรับได้หรือไม่ |
| Accuracy | ตรงกับสูตรหรือแหล่งอ้างอิงหรือไม่ |
| Consistency | ระบบต้นทางต่างกันให้ความหมายเดียวกันหรือไม่ |
| Referential integrity | join กับ dimension ได้ครบหรือไม่ |

### 🧑‍💻 งานที่ 2 — สร้างท่อทำความสะอาด
เขียนฟังก์ชัน `clean(raw)` ที่คืนค่าสามอย่าง: `fact`, `rejects`, `audit`

ข้อกำหนด
* ทุกแถวที่ถูกปฏิเสธต้องอยู่ใน `rejects` พร้อมคอลัมน์ `reject_reason` — **ห้ามทิ้งเงียบๆ**
* ลูกค้าไม่ระบุ **ไม่ใช่** เหตุผลให้ปฏิเสธ ให้ใช้ unknown member (`C-UNKNOWN`) แทน
  เพราะยอดขายยังเป็นยอดขายจริง
* `audit` ต้องบันทึกจำนวนแถวเข้า-ออกของทุกขั้นตอน

*เป้าหมาย: ยอดขายสุทธิสุดท้าย = 3,814,298.55 บาท จาก 3,317 แถว*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 3 — การกระทบยอดและรายงานเหตุผลการปฏิเสธ

### 🧑‍💻 งานที่ 3
1. สรุปจำนวนแถวใน `rejects` แยกตาม `reject_reason` และตาม `source_system`
2. ตรวจว่า `แถวดิบ = แถวใน fact + แถวที่ถูกปฏิเสธ` หรือไม่ (สมการต้องสมดุลเสมอ)
3. คำนวณยอดขายแยกตามระบบต้นทาง แล้วตอบว่าระบบใดได้รับผลกระทบจากการทำความสะอาดมากที่สุด

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **คำถามสำคัญที่ต้องตอบในใบงาน**
>
> ถ้าคุณ **ลืม** เขียนกฎแปลงรูปแบบวันที่ ข้อมูลจาก Marketplace จะถูกปฏิเสธทั้งหมด
> รายงานยอดขายยังออกได้ตามปกติ ไม่มี error ใดๆ แต่ขาดไปทั้งช่องทางการขาย
>
> จงอธิบายว่ากลไกใดในสถาปัตยกรรมคลังข้อมูลที่จะจับความผิดพลาดแบบนี้ได้
> (คำใบ้: ทดลองปิดกฎข้อนี้ในสื่อจำลองแล้วดูว่าตัวเลขใดเปลี่ยน)

### 🧑‍💻 งานที่ 4 — ทดลองปิดกฎทีละข้อ
เขียนโค้ดที่รัน `clean()` ซ้ำโดย **ปิดกฎทีละข้อ** แล้วสร้างตารางเปรียบเทียบว่า
การขาดกฎแต่ละข้อทำให้ยอดขายสุทธิเพี้ยนไปกี่เปอร์เซ็นต์

เรียงลำดับกฎจาก "กระทบตัวเลขมากที่สุด" ไป "น้อยที่สุด"
แล้วตอบว่าถ้ามีเวลาเขียนได้แค่ 3 กฎ ควรเลือกข้อใด เพราะเหตุใด

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **กับดักในการอ่านตารางข้างบน**
>
> กฎ `sku` แสดงความคลาดเคลื่อน **0.00%** — ถ้าอ่านผิวเผินจะสรุปว่า "กฎนี้ไม่สำคัญ"
> ซึ่งเป็นข้อสรุปที่ผิดอย่างอันตราย
>
> การไม่รวมรหัสสินค้าให้เป็นมาตรฐาน **ไม่กระทบยอดรวม** เพราะเงินยังอยู่ครบ
> แต่มันทำลาย **ความสามารถในการวิเคราะห์ระดับสินค้า** ทั้งหมด:
> `P-101`, `P101` และ `101` จะกลายเป็นสินค้าสามชนิดใน `dim_product`
> รายงาน "สินค้าขายดี 10 อันดับ" จะผิด และ join กับตารางสินค้าจะขาดหายไปเงียบๆ
>
> **บทเรียน:** อย่าจัดลำดับความสำคัญของกฎคุณภาพด้วยผลกระทบต่อยอดรวมเพียงอย่างเดียว
> ต้องดูด้วยว่ากฎนั้นปกป้อง **คำถามธุรกิจ** ข้อใด

### 🧑‍💻 งานที่ 5 — พิสูจน์ข้อสังเกตข้างบน
เขียนโค้ดเปรียบเทียบรายงาน "ยอดขาย 5 อันดับแรกตามรหัสสินค้า"
ระหว่างข้อมูลที่ทำ normalize รหัสสินค้าแล้ว กับข้อมูลที่ไม่ได้ทำ
แล้วอธิบายว่าผู้บริหารที่ดูรายงานฉบับหลังจะเข้าใจผิดอย่างไร

In [ ]:
# เขียนโค้ดของคุณที่นี่


---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — profiling ครบ 6 อาการ | 2 |
| งานที่ 2 — ท่อทำความสะอาดที่มี rejects และ audit ครบ | 4 |
| งานที่ 3 — กระทบยอดผ่านและวิเคราะห์ผลกระทบรายระบบต้นทาง | 3 |
| งานที่ 4 — จัดอันดับความสำคัญของกฎพร้อมเหตุผล | 2 |
| งานที่ 5 — พิสูจน์ผลของการไม่ normalize รหัสสินค้า | 2 |
| **รวม** | **13** |

> 💡 ยอดขายสุทธิที่ถูกต้องคือ **3,814,298.55 บาท** จาก **3,317 แถว** —
> ตรงกับตัวเลขบนสื่อจำลองเมื่อเปิดกฎครบทั้ง 8 ข้อ